# DockBench — Inference (Colab / Kaggle)

Chạy trên **Linux** (Colab/Kaggle). Clone repo GitHub → cài molgrid → score **3ERK**.

| Bước | Nội dung |
|------|----------|
| 1 | `git clone` repo |
| 2 | Cài molgrid + PyTorch |
| 3 | `gninatyper` (trong repo hoặc tự cài) |
| 4 | Bảng metrics + load `best_model.pt` |
| 5 | Score 3ERK + xem 3D |

**Runtime:** GPU (khuyến nghị). File `.pt` lớn có thể cần **Git LFS** (`git lfs pull`).

## 1. Cấu hình

In [ ]:
# Repo GitHub (đã push code_docking)
REPO_URL = "https://github.com/nWoWolfpac/KhoaLuanTotNghiep.git"
BRANCH = "main"          # đổi nếu dùng nhánh khác (master, ...)
CLONE_DIR = "/content/KhoaLuanTotNghiep"  # Colab
# CLONE_DIR = "/kaggle/working/KhoaLuanTotNghiep"  # Kaggle: bỏ comment dòng trên, dùng dòng này

MODEL_ID = "geoformerdock"   # gnina_dense | geoformerdock | tankbind | ...
CHECKPOINT = None            # None = lấy best_weights từ summary.json

# gninatyper: ưu tiên tools/ trong repo, hoặc PATH hệ thống
GNINATYPER = None  # None = tự tìm: DOCK_ROOT/tools/gninatyper, rồi which gninatyper

## 2. Clone repository

In [ ]:
import os
import sys
import json
import subprocess
from pathlib import Path

def sh(cmd, check=True):
    print(">>>", cmd)
    return subprocess.run(cmd, shell=True, check=check)

clone = Path(CLONE_DIR)
if not (clone / ".git").is_dir():
    sh(f"git clone --branch {BRANCH} --depth 1 {REPO_URL} {CLONE_DIR}")
else:
    sh(f"cd {CLONE_DIR} && git fetch origin {BRANCH} && git checkout {BRANCH} && git pull --ff-only")

# Thư mục code_docking trong repo
if (clone / "code_docking" / "dockbench").is_dir():
    DOCK_ROOT = clone / "code_docking"
elif (clone / "dockbench").is_dir():
    DOCK_ROOT = clone
else:
    raise FileNotFoundError(f"Không thấy dockbench/ trong {clone}")

sys.path.insert(0, str(DOCK_ROOT))
os.chdir(DOCK_ROOT)
print("DOCK_ROOT =", DOCK_ROOT.resolve())
print("Notebook:", DOCK_ROOT / "DockBench_Inference.ipynb")

In [ ]:
# Git LFS — kéo best_model.pt nếu repo dùng LFS
sh(f"cd {CLONE_DIR} && git lfs install", check=False)
sh(f"cd {CLONE_DIR} && git lfs pull", check=False)

models_root = DOCK_ROOT / "results" / "models"
pts = list(models_root.rglob("best_model.pt")) + list(models_root.rglob("final_model.pt"))
print(f"Tìm thấy {len(pts)} checkpoint .pt trong repo")
for p in pts[:8]:
    print(" ", p.relative_to(DOCK_ROOT), f"({p.stat().st_size / 1e6:.1f} MB)")

## 3. Cài molgrid + PyTorch

In [ ]:
import platform
assert platform.system() == "Linux", "Notebook chỉ chạy trên Colab/Kaggle/Linux."

MF = Path("/usr/local/mambaforge")
if not (MF / "bin/mamba").exists():
    sh("wget -q https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-Linux-x86_64.sh -O /tmp/mf.sh")
    sh("bash /tmp/mf.sh -b -p /usr/local/mambaforge")

sh("/usr/local/mambaforge/bin/mamba install -y -c conda-forge molgrid openbabel 'numpy<2'")
sh("/usr/local/mambaforge/bin/mamba run -n base pip install -q torch py3Dmol pandas")

for pyver in ("3.12", "3.11", "3.10"):
    sp = MF / f"lib/python{pyver}/site-packages"
    if sp.is_dir() and str(sp) not in sys.path:
        sys.path.insert(0, str(sp))

import molgrid
print("molgrid OK")

## 4. gninatyper (PDB → .gninatypes)

Thứ tự tìm: `code_docking/tools/gninatyper` trong repo → `which gninatyper`.

Nếu chưa có: commit binary Linux vào `code_docking/tools/gninatyper`, hoặc build/upload (xem ô comment cuối mục 4).

In [ ]:
import shutil

def find_gninatyper():
    if GNINATYPER is not None:
        p = Path(GNINATYPER)
        if p.is_file():
            return p
    for cand in [
        DOCK_ROOT / "tools" / "gninatyper",
        Path("/content/gninatyper"),
    ]:
        if cand.is_file():
            return cand
    w = shutil.which("gninatyper")
    return Path(w) if w else None

gtyper = find_gninatyper()
if gtyper:
    print("gninatyper:", gtyper)
else:
    print("⚠ Chưa có gninatyper — thêm vào repo tools/ hoặc bật ô build bên dưới.")

In [ ]:
# Tuỳ chọn: build gninatyper (~30–60 phút) — bỏ comment khi cần
# sh("apt-get update -qq && apt-get install -y -qq build-essential cmake git libboost-all-dev libeigen3-dev")
# if not Path("/tmp/gnina/build/bin/gninatyper").exists():
#     sh("git clone --depth 1 --branch v1.3.2 https://github.com/gnina/gnina.git /tmp/gnina")
#     sh("cmake -S /tmp/gnina -B /tmp/gnina/build -DCMAKE_BUILD_TYPE=Release")
#     sh("cmake --build /tmp/gnina/build --target gninatyper -j 2")
# GNINATYPER = Path("/tmp/gnina/build/bin/gninatyper")
# gtyper = find_gninatyper(); print(gtyper)

## 5. Bảng metrics benchmark

In [ ]:
import pandas as pd
from IPython.display import display
from dockbench.models.registry import BENCHMARK_MODELS, MODEL_DISPLAY_NAMES

def load_summary(model_dir: Path):
    p = model_dir / "summary.json"
    return json.loads(p.read_text()) if p.is_file() else {}

rows = []
for mid in BENCHMARK_MODELS:
    s = load_summary(models_root / mid)
    bw = s.get("best_weights", "")
    ck = DOCK_ROOT / bw if bw else None
    ck_ok = ck.is_file() if ck else False
    rows.append({
        "model": MODEL_DISPLAY_NAMES.get(mid, mid),
        "C-index": s.get("final_c_index"),
        "BalAcc": s.get("final_bal_acc"),
        "Pearson": s.get("final_pearson_r"),
        "checkpoint": bw,
        ".pt có": "✓" if ck_ok else ("N/A" if mid == "equibind" else "✗ (LFS?)"),
    })
display(pd.DataFrame(rows))

## 6. Load checkpoint

In [ ]:
import torch
from typing import Optional, Tuple
from dockbench.models.registry import build_model, canonical_name
from dockbench.target_normalizer import TargetNormalizer

def resolve_ckpt(path_str: str) -> Optional[Path]:
    if not path_str:
        return None
    candidates = [
        Path(path_str),
        DOCK_ROOT / path_str,
        DOCK_ROOT / "results" / path_str.replace("\\", "/").split("results/")[-1],
    ]
    for c in candidates:
        if c.is_file():
            return c.resolve()
    return None

def load_model(ckpt_path: Path, model_name: str, input_dims: Tuple[int, int, int, int]):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    payload = torch.load(ckpt_path, map_location=device, weights_only=False)
    name = canonical_name(model_name or payload.get("model", MODEL_ID))
    summary = load_summary(models_root / name)
    gkw = None
    if name == "geoformerdock":
        gkw = {
            "max_pseudo_atoms": int(summary.get("max_pseudo_atoms", 12)),
            "num_transformer_layers": int(summary.get("num_transformer_layers", 2)),
            "uncertainty": bool(summary.get("geoformer_uncertainty", False)),
        }
    model = build_model(name, input_dims, affinity=True, flex=False, geoformer_kwargs=gkw)
    model.load_state_dict(payload["model_state_dict"])
    model.to(device).eval()
    norm = None
    ns = payload.get("target_normalizer")
    if payload.get("normalize_targets") and ns:
        norm = TargetNormalizer()
        norm.mean = float(ns["mean"])
        norm.std = float(ns["std"])
        norm.fitted = True
    return model, device, norm, name

summary = load_summary(models_root / MODEL_ID)
ckpt_path = Path(CHECKPOINT) if CHECKPOINT else resolve_ckpt(summary.get("best_weights", ""))
print("Checkpoint:", ckpt_path)
assert ckpt_path and ckpt_path.is_file(), (
    "Không thấy .pt — đảm bảo đã push file + chạy git lfs pull, hoặc đặt CHECKPOINT=..."
)

## 7. Score 3ERK

In [ ]:
import urllib.request

PDB_ID, LIGAND = "3ERK", "SB4"
WORK = Path("/content/dockbench_work") / PDB_ID
WORK.mkdir(parents=True, exist_ok=True)

pdb = WORK / f"{PDB_ID}.pdb"
if not pdb.is_file():
    urllib.request.urlretrieve(f"http://files.rcsb.org/download/{PDB_ID}.pdb", pdb)

rec, lig = WORK / "rec.pdb", WORK / "lig.pdb"
rec_lines, lig_lines = [], []
for line in pdb.read_text(errors="replace").splitlines(True):
    if line.startswith("ATOM"):
        rec_lines.append(line)
    elif LIGAND in line:
        lig_lines.append(line)
rec.write_text("".join(rec_lines))
lig.write_text("".join(lig_lines))
print(f"rec={len(rec_lines)} lig={len(lig_lines)} atoms")

def pdb_to_gninatypes(pdb_path: Path, out_path: Path):
    exe = find_gninatyper()
    if not exe:
        raise RuntimeError("Thiếu gninatyper — xem mục 4")
    for cmd in (
        [str(exe), str(pdb_path), "-o", str(out_path)],
        [str(exe), "-r", str(pdb_path), "-o", str(out_path)],
        [str(exe), str(pdb_path), str(out_path)],
    ):
        r = subprocess.run(cmd, capture_output=True, text=True)
        if r.returncode == 0 and out_path.is_file() and out_path.stat().st_size > 0:
            return
    raise RuntimeError((r.stderr or r.stdout or "gninatyper failed")[:500])

rec_gt, lig_gt = WORK / "rec.gninatypes", WORK / "lig.gninatypes"
pdb_to_gninatypes(rec, rec_gt)
pdb_to_gninatypes(lig, lig_gt)
(WORK / "score.types").write_text(f"1 0.0 {rec_gt.name} {lig_gt.name}\n")
print("gninatypes OK")

In [ ]:
gmaker = molgrid.GridMaker(resolution=0.5, dimension=23.5)
prov = molgrid.ExampleProvider(
    data_root=str(WORK),
    balanced=False,
    shuffle=False,
    default_batch_size=1,
    iteration_scheme=molgrid.IterationScheme.SmallEpoch,
    cache_structs=False,
)
prov.populate(str(WORK / "score.types"))
INPUT_DIMS = tuple(int(x) for x in gmaker.grid_dimensions(prov.num_types()))
print("input_dims (C,D,H,W) =", INPUT_DIMS)

model, device, normalizer, model_name = load_model(ckpt_path, MODEL_ID, INPUT_DIMS)
batch = prov.next_batch(1)
grid = torch.zeros((1,) + INPUT_DIMS, dtype=torch.float32, device=device)
gmaker.forward(batch, grid, random_translation=0.0, random_rotation=False)

with torch.no_grad():
    pose_log, aff = model(grid)

pose_prob = float(torch.exp(pose_log)[0, 1].item())
aff_val = float(aff[0].item())
if normalizer and normalizer.fitted:
    aff_val = float(normalizer.denormalize(aff)[0].item())

print(f"Model: {model_name}")
print(f"Pose P(good): {pose_prob:.4f}  → label {int(pose_prob >= 0.5)}")
print(f"Affinity (pK pred): {aff_val:.4f}")

## 8. Xem 3D (py3Dmol)

In [ ]:
import py3Dmol
view = py3Dmol.view(width=700, height=450)
view.addModel(rec.read_text(), "pdb")
view.setStyle({"model": 0}, {"cartoon": {"color": "spectrum"}})
view.addModel(lig.read_text(), "pdb")
view.setStyle({"model": 1}, {"stick": {"colorscheme": "greenCarbon"}})
view.zoomTo()
view.show()

## 9. (Tuỳ chọn) Batch inference từ `.types`

Cần thư mục `data/PDBbind2016/` (hoặc data_root tương ứng) đã có trong repo hoặc tải thêm.

In [ ]:
# TYPES_FILE = DOCK_ROOT / "data" / "types" / "ref_uff_test0.types"
# DATA_ROOT = DOCK_ROOT / "data" / "PDBbind2016"
# from dockbench.dataloaders import GriddedExamplesLoader
# from dockbench.setup import setup_example_provider, setup_grid_maker
# import argparse
# args = argparse.Namespace(
#     data_root=str(DATA_ROOT), batch_size=32, shuffle=False,
#     iteration_scheme="small", ligmolcache="", recmolcache="",
#     stratify_receptor=False, stratify_pos=0, stratify_max=0, stratify_min=0, stratify_step=0,
#     cache_structures=False, dimension=23.5, resolution=0.5,
# )
# provider = setup_example_provider(str(TYPES_FILE), args, training=False)
# grid_maker = setup_grid_maker(args)
# loader = GriddedExamplesLoader(
#     provider, grid_maker, label_pos=0, affinity_pos=1, rmsd_pos=None,
#     random_translation=0.0, random_rotation=False, device=device,
# )
# print("Batch dims", loader.dims)